# SAM 3 labels, YOLO26 vs RF-DETR bake-off, and Moondream 2 vs 3.1 on bumpers

One notebook, one dataset, one GPU, one metric. Run the cells top to bottom.

**Budget (Colab Pro, 200 compute units/month).** Rates are approximate; Colab shows the exact rate under *Runtime → Change runtime type*.

| GPU | ~units/hour | use it for |
|---|---|---|
| T4 | 1.5–2 | the dry run (`DRY_RUN = True`), Moondream 2 |
| **L4** | 4–5 | **the real run** and Moondream 3.1 |
| A100 | 11–13 | only if the L4 runs out of memory at 960 px |

A real run is about 3–8 h per model on an L4, roughly 40–75 units for both plus the bumper test.

**Before you start**, put this in Google Drive under `MyDrive/frc-bakeoff/`:

```
frc-bakeoff/
  videos/
    e1.mp4  e4.mp4  e5.mp4  e8.mp4   # the 2026 Einstein broadcasts (1080p); the dataset is rebuilt from these
```

The dataset is rebuilt the way `fuel_relabel.pt` / `scout_relabel.pt` were trained (`train/relabel_video.py`, settings from `train/README.md`), with the original labelling models downloaded from the v0.3.0 release and checked against their recorded checksums. It is saved to Drive as `dataset-relabel.zip`, so it is built once. Have a dataset zip instead? Set `SOURCE = "zip"` and put it in `frc-bakeoff/` as `dataset-scout.zip`.

Everything the run writes (weights, checkpoints, results) goes to `frc-bakeoff/runs/` in Drive, so a disconnect loses nothing: re-run the cells and training resumes.

**Do the dry run first on a T4** (`DRY_RUN = True`: 1 epoch on a few images, ~10 min). It catches a bad zip or path before you spend L4 hours. Then switch the runtime to L4, set `DRY_RUN = False` and run everything again.

## 1. Settings

In [ ]:
DRY_RUN = False           # True: 1 epoch on a few images to test the setup. False: the real run (dry run passed 2026-11-16).

DRIVE_DIR   = "/content/drive/MyDrive/frc-bakeoff"
SOURCE      = "einstein"   # "einstein": rebuild the dataset from Einstein videos in DRIVE_DIR/videos
                           # "zip": use a dataset zip you uploaded (DATASET_ZIP)
DATASET_ZIP = "dataset-scout.zip"     # in DRIVE_DIR, only for SOURCE = "zip"
# Einstein video file -> "split:start", start = match-clock start in video seconds (train/README.md)
EINSTEIN = {"e4.mp4": "train:6", "e5.mp4": "train:6", "e8.mp4": "train:7", "e1.mp4": "val:6"}
BRANCH      = "claude/practical-einstein-bgrb3r"   # switch to "main" once this work is merged

IMGSZ       = 960         # fuel is ~17x12 px; 960 is what train.py uses. RF-DETR needs a multiple of 32.
EPOCHS      = 50
YOLO_MODEL  = "yolo26s.pt"
RFDETR_SIZE = "small"     # nano / small / medium
BATCH       = 8           # lower to 4 if either model runs out of GPU memory

# SAM 3 labels (section 4b). Needs approved access to facebook/sam3 on Hugging Face and
# a read token saved in Colab Secrets (key icon, left) as HF_TOKEN.
SAM3_COMPARE = 0          # draw current vs SAM 3 labels for this many frames; 0 skips (12 were checked 2026-11-16)
USE_SAM3     = True       # True: SAM 3 fuel + current robots, train both models on that (chosen 2026-11-16)
FUEL_PROMPT  = "yellow ball"   # next season: describe the new game piece

# Bumper test on a real match (optional). Team numbers from TBA's match page.
BUMPER_VIDEO = ""         # e.g. "videos/qm7.mp4" (relative to DRIVE_DIR); "" skips it
RED_TEAMS    = ""         # e.g. "1058,611,8013"
BLUE_TEAMS   = ""

## 2. GPU check

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 / L4 / A100"
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{name}: compute capability {major}.{minor}, {mem:.0f} GB")
CAN_RUN_MD31 = major >= 8     # Moondream 3.1's Photon engine needs Ampere or newer (L4, A100: yes; T4: no)
print("Moondream 3.1 can run here:", CAN_RUN_MD31)
if not DRY_RUN and "T4" in name:
    print("!! Real run on a T4: it will work but slowly. An L4 is about 3x faster for ~2.5x the units.")

## 3. Drive, code, packages

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
if SOURCE == "zip":
    assert os.path.exists(f"{DRIVE_DIR}/{DATASET_ZIP}"), f"put {DATASET_ZIP} in {DRIVE_DIR} first"

In [ ]:
%cd /content
!rm -rf frc && git clone -q --depth 1 -b {BRANCH} https://github.com/ShadowOfTheVOID/YOLOv26-FRC-Model frc
%cd /content/frc
# runs/ lives in Drive: weights and checkpoints survive a disconnect
!rm -rf runs && ln -s "{DRIVE_DIR}/runs" runs
# one set of versions for everything: RF-DETR 1.12+, SAM 3 and Moondream 2 all run on transformers 5
!pip -q install "ultralytics>=8.4" "rfdetr[train]>=1.12" "transformers>=5.1"

## 4. Dataset

With `SOURCE = "einstein"` (the default) the first cell checks your videos and rebuilds the dataset; with `SOURCE = "zip"` the second cell unzips yours. Either way the dataset is copied to the local disk: reading thousands of small files straight from Drive is slow enough to starve the GPU.

In [ ]:
import glob, shutil, random
if SOURCE == "zip":
    !rm -rf /content/ds && mkdir -p /content/ds && unzip -q "{DRIVE_DIR}/{DATASET_ZIP}" -d /content/ds
    found = glob.glob("/content/ds/**/dataset.yaml", recursive=True)
    assert found, "no dataset.yaml inside the zip"
    DATA = found[0]
    root = pathlib.Path(DATA).parent
    for split in ("train", "val"):
        n = len(list((root / "images" / split).glob("*")))
        print(split, n, "images")

    if DRY_RUN:   # a few images per split, so the dry run takes minutes
        small = pathlib.Path("/content/ds_dry")
        shutil.rmtree(small, ignore_errors=True)
        for split, k in (("train", 16), ("val", 8)):
            imgs = sorted((root / "images" / split).glob("*"))
            random.Random(0).shuffle(imgs)
            for kind in ("images", "labels"):
                (small / kind / split).mkdir(parents=True)
            for p in imgs[:k]:
                shutil.copy(p, small / "images" / split)
                lab = root / "labels" / split / (p.stem + ".txt")
                if lab.exists():
                    shutil.copy(lab, small / "labels" / split)
        shutil.copy(DATA, small / "dataset.yaml")
        DATA = str(small / "dataset.yaml")
    print("training on", DATA)

In [ ]:
import glob, shutil, random, hashlib, subprocess, json as _json
if SOURCE == "einstein":
    # 1. Which videos are there, and are they the broadcasts the settings were made for?
    vdir = f"{DRIVE_DIR}/videos"
    found = {}
    for name, spec in EINSTEIN.items():
        path = f"{vdir}/{name}"
        if not os.path.exists(path):
            print(f"  missing  {name}  ({spec})"); continue
        info = _json.loads(subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                                           "stream=width,height,r_frame_rate:format=duration", "-of", "json", path],
                                          capture_output=True, text=True).stdout or "{}")
        st = (info.get("streams") or [{}])[0]; dur = float(info.get("format", {}).get("duration", 0))
        fps = eval(st.get("r_frame_rate", "0/1")) if st else 0
        ok = st.get("width") == 1920 and st.get("height") == 1080
        print(f"  found    {name}  ({spec})  {st.get('width')}x{st.get('height')}  {fps:.0f} fps  {dur/60:.1f} min"
              + ("" if ok else "   !! not 1920x1080: the split-screen rows and masks below are for 1080p Einstein broadcasts"))
        found[name] = spec
    splits = {s.split(":")[0] for s in found.values()}
    if not {"train", "val"} <= splits:
        raise SystemExit(f"Need at least one train and one val video in {vdir} (have: {sorted(found) or 'none'}). "
                         "Rename them to match EINSTEIN in the settings cell, or change EINSTEIN.")
    print(f"{len(found)} of {len(EINSTEIN)} Einstein videos found -- enough to rebuild.")

In [ ]:
if SOURCE == "einstein":
    zip_name = "dataset-relabel-dry.zip" if DRY_RUN else "dataset-relabel.zip"
    out = "/content/ds_relabel"
    if os.path.exists(f"{DRIVE_DIR}/{zip_name}"):
        print("reusing", zip_name, "from Drive (delete it there to rebuild)")
        !rm -rf {out} && mkdir -p {out} && unzip -q "{DRIVE_DIR}/{zip_name}" -d {out}
    else:
        # 2. The labelling models that built the original set, checked against CHANGELOG's checksums
        SHA = {"fuel_best.pt": "202429d3e8a7640e444e8053b229998b363f698469274829a069dd015994b5f3",
               "fuel_withBotbest.pt": "0c5267368787e054f9fbd57c49078726306ca3370b7185752c3c2229dfffd14e"}
        os.makedirs("/content/models", exist_ok=True)
        for m, want in SHA.items():
            dst = f"/content/models/{m}"
            if not os.path.exists(dst):
                !curl -sL -o {dst} https://github.com/ShadowOfTheVOID/YOLOv26-FRC-Model/releases/download/v0.3.0/{m}
            got = hashlib.sha256(open(dst, "rb").read()).hexdigest()
            assert got == want, f"{m}: checksum {got[:12]}... is not the recorded one -- not the original model"
        # 3. relabel_video.py with train/README.md's Einstein settings
        vids = " ".join(f'--video "{DRIVE_DIR}/videos/{n}:{s}"' for n, s in found.items())
        length = "--length 15" if DRY_RUN else ""
        !rm -rf {out} && python train/relabel_video.py --out {out} {vids} {length} --rows 0:700 \
            --mask 440,0,1480,165 --mask 15,58,440,122 --mask 1480,58,1905,122 \
            --fuel /content/models/fuel_best.pt --robots /content/models/fuel_withBotbest.pt --device 0
        if globals().get("_exit_code", 0) != 0:   # stop on relabel_video's own error, not on the empty set it left behind
            raise SystemExit("relabel_video.py failed: its error is just above this message.")
        if all(glob.glob(f"{out}/images/{s}/*") for s in ("train", "val")):   # never save an empty set for reuse
            !cd {out} && zip -qr "{DRIVE_DIR}/{zip_name}" . -x "preview/*" && echo "saved {zip_name} to Drive"
    DATA = f"{out}/dataset.yaml"
    counts = {split: len(glob.glob(f"{out}/images/{split}/*")) for split in ("train", "val")}
    print(counts)
    if not all(counts.values()):
        # e.g. a start time past the end of a short or different video: training on nothing would "succeed" silently
        bad = [s for s, c in counts.items() if not c]
        raise SystemExit(f"No {' or '.join(bad)} images were made. Check the videos are the full Einstein broadcasts "
                         f"and the start times in EINSTEIN, then re-run.")
    # 4. Look before training (train/README.md): green = detector, cyan = detector confirmed by colour, magenta = added by motion, orange/red = robots; grey patches are painted-out yellow
    from IPython.display import Image as Show, display
    for f in sorted(glob.glob(f"{out}/preview/*.jpg"))[:4]:
        print(f); display(Show(f, width=1100))
    print("training on", DATA)

## 4b. SAM 3 labels (optional)

SAM 3 finds every instance of a text prompt, so balls in piles, in hoppers and in the air should get boxes where `autolabel_fuel`'s colour rules gave up. **This is not measured on our frames yet**, so look before trusting it:

1. With `SAM3_COMPARE = 12`, the first cell draws 12 random frames: current labels on top, SAM 3 below. Magenta = a pile SAM 3 returned as one box (that frame would be left out). SAM 3 boxes **fuel only**; robots are copied from the current labels (the scouting model's, checked good by eye), because on the first comparison SAM 3 found 0-1 of ~4 robots per frame.
2. If SAM 3 looks better, set `USE_SAM3 = True` and run the second cell. It labels the whole dataset into a new one (the original is untouched), saves it to Drive as `dataset-sam3.zip`, and the training cells below then use it.

With `SOURCE = "einstein"`, SAM 3 sees the frames re-cut from your videos **without** the grey paint the rebuild puts over yellow it could not box (those are the hard balls SAM 3 is being tried for); the scoreboard masks stay. In the comparison, yellow left without a box in the top half is what the current labels missed.

Hub labels are copied from the current labels: they came from geometry recorded by hand, which beats any prompt.

Every model in this notebook (SAM 3, RF-DETR 1.12+, Moondream 2) runs on the same `transformers` 5 installed in section 3.

In [ ]:
# SAM 3 runs only to draw the comparison or to label a dataset not yet saved in Drive. Resuming
# with dataset-sam3.zip already there needs neither the token nor the unpainted frames.
_sam3_zip = f"{DRIVE_DIR}/" + ("dataset-sam3-dry.zip" if DRY_RUN else "dataset-sam3.zip")
SAM3_NEEDED = bool(SAM3_COMPARE) or (USE_SAM3 and not os.path.exists(_sam3_zip))
if SAM3_NEEDED:
    from google.colab import userdata
    for _name in ("HF_TOKEN", "HF_Token", "hf_token"):   # Colab secret names are case-sensitive
        try:
            os.environ["HF_TOKEN"] = userdata.get(_name)
            break
        except Exception:
            pass
    else:
        # No Colab Secret (the phone layout hides the Secrets panel): ask instead. getpass hides
        # the token as it is pasted and it is kept only in this session, never in the notebook.
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("Hugging Face read token (huggingface.co -> Settings -> Access Tokens): ").strip()
    if not os.environ.get("HF_TOKEN"):
        raise SystemExit("No Hugging Face token: add HF_TOKEN in Colab Secrets or paste it when asked "
                         "(and request access at huggingface.co/facebook/sam3).")
# SAM 3 works on the frames WITHOUT relabel_video's grey paint: the paint covers exactly the hard
# balls (piles, hoppers, in flight) SAM 3 is being tried for (train/unpaint_relabel.py).
SAM_DATA = DATA
if SAM3_NEEDED and SOURCE == "einstein":
    if not os.path.exists("/content/ds_unpainted/dataset.yaml"):
        !python train/unpaint_relabel.py --data {DATA} --videos "{DRIVE_DIR}/videos" --out /content/ds_unpainted
    SAM_DATA = "/content/ds_unpainted/dataset.yaml"
if SAM3_COMPARE:
    n = min(SAM3_COMPARE, 3) if DRY_RUN else SAM3_COMPARE
    !python train/autolabel_sam3.py --data {SAM_DATA} --compare {n} --preview /content/sam3_compare --fuel-prompt "{FUEL_PROMPT}"
    !mkdir -p "{DRIVE_DIR}/runs/sam3_compare" && cp /content/sam3_compare/*.jpg "{DRIVE_DIR}/runs/sam3_compare/"
    from IPython.display import Image as Show, display
    for f in sorted(glob.glob("/content/sam3_compare/*.jpg"))[:6]:
        print(f); display(Show(f, width=1100))

In [ ]:
if USE_SAM3:
    zip_name = "dataset-sam3-dry.zip" if DRY_RUN else "dataset-sam3.zip"
    out = "/content/ds_sam3"
    if os.path.exists(f"{DRIVE_DIR}/{zip_name}"):
        print("reusing", zip_name, "from Drive (delete it there to relabel)")
        !rm -rf {out} && mkdir -p {out} && unzip -q "{DRIVE_DIR}/{zip_name}" -d {out}
    else:
        limit = "--limit 8" if DRY_RUN else ""
        !python train/autolabel_sam3.py --data {SAM_DATA} --out {out} --fuel-prompt "{FUEL_PROMPT}" {limit}
        # zip stores the files the image links point to, so the zip stands alone in Drive
        !cd {out} && zip -qr "{DRIVE_DIR}/{zip_name}" . && echo "saved {zip_name} to Drive"
    DATA = f"{out}/dataset.yaml"
    print("training on SAM 3 labels:", DATA)
    !cat {out}/skipped.txt | cut -f2 | sort | uniq -c | sort -rn | head

In [ ]:
TAG = ("dry" if DRY_RUN else "bake") + ("_sam3" if USE_SAM3 else "")
EP = 1 if DRY_RUN else EPOCHS
YOLO_NAME, RF_NAME = f"yolo_{TAG}", f"rfdetr_{TAG}"

def fresh(out, *deps):
    """True when `out` exists and is newer than every existing dep: a result already in Drive
    that nothing it was made from has changed since. Re-running it would only spend GPU time."""
    o = pathlib.Path(out)
    if not o.exists():
        return False
    ds = [pathlib.Path(d) for d in deps if pathlib.Path(d).exists()]
    return all(o.stat().st_mtime > d.stat().st_mtime for d in ds)


## 5. Train YOLO26
If this run was cut off, re-running the cell resumes from `last.pt` in Drive.

In [ ]:
run = pathlib.Path(f"runs/{YOLO_NAME}")
last = run / "weights" / "last.pt"
if (run / "DONE").exists():
    print("already trained:", run / "weights" / "best.pt", "(delete DONE in Drive to train again)")
elif last.exists():
    print("resuming", last)
    !python train/train.py --model {last} --resume --data {DATA} --name {YOLO_NAME} && touch {run}/DONE
else:
    !python train/train.py --data {DATA} --model {YOLO_MODEL} --imgsz {IMGSZ} --epochs {EP} --batch {BATCH} --workers 4 --name {YOLO_NAME} && touch {run}/DONE

## 6. Train RF-DETR
Same data, same resolution. Re-running the cell resumes from the last checkpoint in Drive (`last.ckpt`; `checkpoint.pth` from RF-DETR before 1.7).

In [ ]:
run = pathlib.Path(f"runs/{RF_NAME}")
ckpt = next((run / f for f in ("last.ckpt", "checkpoint.pth") if (run / f).exists()), run / "last.ckpt")
if (run / "DONE").exists():
    print("already trained:", run / "checkpoint_best_total.pth", "(delete DONE in Drive to train again)")
else:
    resume = f"--resume {ckpt}" if ckpt.exists() else ""
    if resume: print("resuming", ckpt)
    !python train/train_rfdetr.py --data {DATA} --size {RFDETR_SIZE} --imgsz {IMGSZ} --epochs {EP} --batch {max(1, BATCH // 2)} --grad-accum 4 --workers 8 --name {RF_NAME} {resume} && touch {run}/DONE

## 7. Compare on the same val images with the same metric

`AP50sm` (boxes under 24 px) is the column that decides fuel. All numbers are agreement with the auto-labels, so look at disagreements by eye before believing a small gap.

With `USE_SAM3 = True` both models are trained **and scored** on SAM 3's labels, so these numbers measure agreement with SAM 3. To see whether SAM 3 labels made the detector better at the real thing, compare on hand-checked frames or on hand counts, not on this table.


In [ ]:
out = f"runs/compare_{TAG}.txt"
if fresh(out, f"runs/{YOLO_NAME}/DONE", f"runs/{RF_NAME}/DONE"):
    print(open(out).read()); print("(already scored; delete", out, "in Drive to score again)")
else:
    !python train/compare_detectors.py --data {DATA} --imgsz {IMGSZ} \
        --yolo runs/{YOLO_NAME}/weights/best.pt \
        --rfdetr runs/{RF_NAME}/checkpoint_best_total.pth --rfdetr-size {RFDETR_SIZE} | tee {out}
    print("saved to Drive:", out)


## 8. Bumpers: Moondream 2 vs 3.1 on the same crops

The size test from `experiments/bumper_moondream.py`: one public 2024 field photo (Wikimedia Commons), each robot shrunk until its numerals are 35 → 7 px tall. On a 1080p wide broadcast shot they are about 10 px. Moondream 2 on CPU got 360 right down to 7 px, read 3268 as "32 69" at 14 px, and made up "131" for a half-hidden bumper. The question is whether 3.1 does better at 10–14 px.

In [ ]:
BUMPERS_DONE = os.path.exists("runs/bumpers_md2.txt") and (os.path.exists("runs/bumpers_md31.txt") or not CAN_RUN_MD31)
if BUMPERS_DONE:
    print(open("runs/bumpers_md2.txt").read()); print("(already run; delete runs/bumpers_md2.txt in Drive to run again)")
else:
    import urllib.request
    PHOTO = "/content/crescendo.jpg"
    URL = "https://upload.wikimedia.org/wikipedia/commons/6/60/FIRST_Robotics_Competition_%28FRC%29_2024_Crescendo_Playing_Field.jpg"
    if os.path.exists(f"{DRIVE_DIR}/crescendo.jpg"):
        shutil.copy(f"{DRIVE_DIR}/crescendo.jpg", PHOTO)
    elif not os.path.exists(PHOTO):
        try:
            req = urllib.request.Request(URL, headers={"User-Agent": "frc-bakeoff-notebook/0.1"})
            open(PHOTO, "wb").write(urllib.request.urlopen(req, timeout=60).read())
        except Exception as e:
            raise SystemExit(f"Could not download the test photo ({e}). Download it in a browser from\n{URL}\n"
                             f"and put it in Drive as {DRIVE_DIR}/crescendo.jpg, then re-run this cell.")
    # boxes are for the 3840 px wide version the table was measured on
    from PIL import Image
    im = Image.open(PHOTO)
    if im.width != 3840:
        im.resize((3840, round(im.height * 3840 / im.width)), Image.LANCZOS).save(PHOTO)
    BOXES = "--box 2130,716,2407,926=360 --box 2010,780,2165,895=3268 --box 3331,768,3601,933="
    !python experiments/bumper_moondream.py --moondream 2 --scale-test {PHOTO} {BOXES} | tee runs/bumpers_md2.txt


In [ ]:
if BUMPERS_DONE:
    pass
elif CAN_RUN_MD31:
    !pip -q install moondream
    !python experiments/bumper_moondream.py --moondream 3.1 --scale-test {PHOTO} {BOXES} | tee runs/bumpers_md31.txt
else:
    print("Moondream 3.1 needs an Ampere or newer GPU (L4 / A100). Switch runtime to run this cell.")

## 9. Bumpers on a real match (optional)

Reads every robot box once a second with the better detector and writes a CSV to Drive. Answers are only accepted when they name a team on that alliance (exact or one digit off). Open the CSV and check a sample by eye: right team, wrong team, or blank.

In [ ]:
if BUMPER_VIDEO and not DRY_RUN:
    weights = f"runs/{YOLO_NAME}/weights/best.pt"   # or runs/{RF_NAME}/checkpoint_best_total.pth if RF-DETR won
    version = "3.1" if CAN_RUN_MD31 else "2"
    !python experiments/bumper_moondream.py --moondream {version} --video "{DRIVE_DIR}/{BUMPER_VIDEO}" \
        --weights {weights} --red {RED_TEAMS} --blue {BLUE_TEAMS} --every 1.0 --out runs/bumper_reads.csv
else:
    print("skipped: set BUMPER_VIDEO (and DRY_RUN = False)")

## 11. Experiments: YOLO26 variants, judged by counting Einstein 1

On the SAM 3 labels YOLO26 and RF-DETR tied at ~0.94 fuel AP50: the score table can no longer tell models apart. So every model here is also run through the **shipped hub counter on Einstein 1** (a match none of them trained on) and scored against the scoreboard at the buzzer (`train/count_einstein.py`). That is the number that matters; the current `fuel_relabel.pt` is the baseline to beat.

| variant | why | ~L4 time |
|---|---|---|
| `yolo26s_p2` | extra detection layer for tiny objects (fuel ~15 px) | 1.5 h |
| `yolo26m` | a bigger model of the same kind | 1.5 h |
| `yolo26s_1280` | small balls stay bigger (the MI300X crash was P2 *with* 1280) | 2 h |
| `yolo26n` | the fastest, for live counting on a laptop CPU | 40 min |

Plus one **tuned RF-DETR** (`rfdetr_q900_cos`, ~2 h): the first run could report at most 300 objects per image while Einstein frames average ~170 fuel and the crowded ones pass 300, and its learning rate never decayed (the default step only drops at epoch 100). This one holds 900 objects, uses a cosine schedule, and stops at 30 epochs (it peaked at 27). RF-DETR is scored in the table, which now also gives each model's **best F1 at its own threshold**; it cannot run in the hub counter yet (that loads YOLO models only), so it gets no counting row.

Each trains on the same dataset as above, is skipped once done, and resumes after a disconnect like the rest. Remove a line from `VARIANTS` to skip it. `QUICK = True` (the default) runs only `yolo26s_p2` and `yolo26n`, about 1.5 h; set it to `False` for the rest. Everything: roughly 4-5 h on an L4 (about half that on an A100, for about the same units) -- 25 epochs, images cached in RAM and an automatic batch size; not yet timed for real.

In [ ]:
RUN_EXPERIMENTS = True
EXP_EPOCHS = 25        # YOLO26s peaked at epochs 10-25 on this data (2026-11-16); best.pt keeps the best epoch
FAST = "--cache ram --workers 8"   # decoded images in RAM: JPEG decoding on few CPU cores starves the GPU
# name: (starting weights, image size, P2 head, batch); 0.8 = fill 80% of GPU memory (Ultralytics autobatch)
VARIANTS = {
    "yolo26s_p2":   ("yolo26s.pt", 960,  True,  0.8),
    "yolo26m":      ("yolo26m.pt", 960,  False, 0.8),
    "yolo26s_1280": ("yolo26s.pt", 1280, False, 0.8),
    "yolo26n":      ("yolo26n.pt", 960,  False, 0.8),
}
# tuned RF-DETR: name -> (object queries, schedule, epochs); empty {} skips it
RFDETR_TUNED = {"rfdetr_q900_cos": (900, "cosine", 30)}

# QUICK: only the two most telling runs, ~1.5 h on an L4 (small-object head and the fast model),
# then the score table and the Einstein 1 counting. The rest can run another day: finished
# runs are kept in Drive and skipped, so setting QUICK = False later only adds what is missing.
QUICK = True
if QUICK:
    VARIANTS = {k: VARIANTS[k] for k in ("yolo26s_p2", "yolo26n")}
    RFDETR_TUNED = {}
print("experiments:", list(VARIANTS) + list(RFDETR_TUNED))


In [ ]:
if RUN_EXPERIMENTS:
    ep = 1 if DRY_RUN else EXP_EPOCHS
    for v, (start, size, p2, batch) in VARIANTS.items():
        name = f"{v}_{TAG}"
        run = pathlib.Path(f"runs/{name}")
        last = run / "weights" / "last.pt"
        if (run / "DONE").exists():
            print(f"{name}: already trained"); continue
        print(f"===== {name} =====", flush=True)
        if last.exists():
            !python train/train.py --model {last} --resume --data {DATA} --name {name} && touch {run}/DONE
        else:
            p2flag = "--p2" if p2 else ""
            !python train/train.py --data {DATA} --model {start} {p2flag} --imgsz {size} --epochs {ep} --batch {batch} {FAST} --name {name} && touch {run}/DONE
        if not (run / "DONE").exists():
            print(f"!! {name} did not finish -- its error is above; the next variant still runs. "
                  f"(Out of memory: set its batch in VARIANTS to a whole number such as 4 and re-run.)")

In [ ]:
if RUN_EXPERIMENTS:
    for v, (queries, sched, epochs) in RFDETR_TUNED.items():
        name = f"{v}_{TAG}"
        run = pathlib.Path(f"runs/{name}")
        if (run / "DONE").exists():
            print(f"{name}: already trained"); continue
        print(f"===== {name} =====", flush=True)
        ckpt = next((run / f for f in ("last.ckpt", "checkpoint.pth") if (run / f).exists()), None)
        resume = f"--resume {ckpt}" if ckpt else ""
        ep = 1 if DRY_RUN else epochs
        !python train/train_rfdetr.py --data {DATA} --size {RFDETR_SIZE} --imgsz {IMGSZ} --epochs {ep} --batch {max(1, BATCH // 2)} --grad-accum 4 --workers 8 --name {name} --queries {queries} --scheduler {sched} {resume} && touch {run}/DONE
        if not (run / "DONE").exists():
            print(f"!! {name} did not finish -- its error is above. (Out of memory: lower BATCH, or queries to 600.)")

In [ ]:
import hashlib
if RUN_EXPERIMENTS:
    # 1. the same score table as section 7, every model at once
    yolos = [f"--yolo base=runs/{YOLO_NAME}/weights/best.pt"]
    for v, (_, size, _, _) in VARIANTS.items():
        yolos.append(f"--yolo {v}=runs/{v}_{TAG}/weights/best.pt@{size}")
    detrs = [f"--rfdetr rfdetr_base=runs/{RF_NAME}/checkpoint_best_total.pth"]
    detrs += [f"--rfdetr {v}=runs/{v}_{TAG}/checkpoint_best_total.pth" for v in RFDETR_TUNED]
    yolos += detrs
    out = f"runs/compare_experiments_{TAG}.txt"
    dones = [f"runs/{YOLO_NAME}/DONE", f"runs/{RF_NAME}/DONE"] + \
            [f"runs/{v}_{TAG}/DONE" for v in list(VARIANTS) + list(RFDETR_TUNED)]
    if fresh(out, *dones):
        print(open(out).read()); print("(already scored; delete it in Drive to score again)")
    else:
        !python train/compare_detectors.py --data {DATA} --imgsz {IMGSZ} {" ".join(yolos)} | tee {out}

    # 2. what matters: counting Einstein 1 against its scoreboard, current model as the baseline
    cur = "/content/models/fuel_relabel.pt"
    if not os.path.exists(cur):
        !mkdir -p /content/models && curl -sL -o {cur} https://github.com/ShadowOfTheVOID/YOLOv26-FRC-Model/releases/download/v0.5.8/fuel_relabel.pt
    assert hashlib.sha256(open(cur, "rb").read()).hexdigest() == \
        "bf9eb2c3cb25b119b54bd9a09d35227ad30a99a371a5dc532828cd9c0d7f4977", "fuel_relabel.pt is not the released one"
    models = [f"--model current={cur}", f"--model base_sam3=runs/{YOLO_NAME}/weights/best.pt"]
    models += [f"--model {v}=runs/{v}_{TAG}/weights/best.pt" for v in VARIANTS]
    end = "--end 40" if DRY_RUN else ""
    out = f"runs/count_e1_{TAG}.txt"
    if fresh(out, *dones):
        print(open(out).read()); print("(already counted; delete it in Drive to count again)")
    else:
        !python train/count_einstein.py --video "{DRIVE_DIR}/videos/e1.mp4" {" ".join(models)} {end} --out {out}
    print("saved to Drive:", out)

## 10. Stop paying
An idle GPU session keeps using compute units. Everything worth keeping is already in Drive.

In [ ]:
from google.colab import runtime
runtime.unassign()